# From PDF Files to a Raw Text Corpus

Political manifestos are usually distributed as **PDF documents designed for people to read**.  
Text analytics requires a different representation: we need to extract the text and organize the documents so that Python can process them systematically.

In this notebook we work only on the **EXTRACT** stage of the text-analytics journey.

We will:

1. locate the manifesto PDFs stored in the `RAW` folder of a GitHub repository;
2. identify the direct URL of each PDF;
3. read every PDF and extract its text page by page;
4. organize the documents into a Pandas DataFrame;
5. calculate a few simple document-size indicators; and
6. save the resulting corpus as `corpus_raw.csv`.

> **Important:** We do **not** clean the text in this notebook.  
> No lowercasing, stopword removal, tokenization, lemmatization, stemming, punctuation removal, or n-grams are applied here.

Our final rule will be:

> **One row = one party manifesto.**

---

## How AI can help in this notebook

For an intermediate computational user, AI assistance is usually **GOOD for coding** at this stage.  
However, the amount of help a student needs depends on their programming experience.

Throughout the notebook, optional prompts are provided at three levels:

- **Novice:** asks AI to explain the idea and produce explicit code.
- **Intermediate:** asks AI for a concise implementation while preserving readability.
- **Advanced:** asks AI to propose alternatives, robustness checks, or more scalable solutions.

You do **not** need to use every prompt. They are examples of how the same computational task can be communicated to AI at different levels of expertise.


## 1. Locate the source files

The manifesto PDFs are stored in the `RAW` folder of a public GitHub repository.

We could manually type the name and URL of every PDF, but that would make the notebook fragile.  
Instead, we ask the **GitHub API** to tell us which files are currently inside that folder.

The API returns metadata for each file, including:

- its filename;
- its type;
- its location in the repository; and
- a direct `download_url`.

This means the notebook can **discover the documents programmatically**.

### Optional AI prompts

**Novice**

> I have PDF files in the `RAW` folder of a public GitHub repository. I want Python code that asks the GitHub API which files are in that folder. Please keep the code simple and explain what each line does.

**Intermediate**

> Using Python `requests`, query the GitHub Contents API for the `RAW` folder of `eScience-SummerSchool/manifestos` and return the metadata for the files in that folder.

**Advanced**

> Show me two reproducible ways to discover files stored in a public GitHub directory without cloning the repository. Compare the GitHub Contents API with other alternatives and explain the trade-offs.


In [ ]:
import requests

api_url = "https://api.github.com/repos/eScience-SummerSchool/manifestos/contents/RAW"

files = requests.get(api_url).json()

files


### What did we get?

`files` is a Python object containing metadata returned by GitHub.

At this point we have **not read any manifesto yet**.  
We have only discovered what is available in the source folder.

This is a useful checkpoint: before processing data, first verify what the source actually contains.


## 2. Keep only the PDF files

A repository folder can contain many kinds of files.  
Our corpus should include only the political manifestos stored as PDFs.

We therefore filter the GitHub metadata and retain files whose names end in `.pdf`.

After filtering, we print the filenames to verify exactly which documents will enter the corpus.

### Optional AI prompts

**Novice**

> I have a Python list of dictionaries returned by the GitHub API. Each dictionary has a `"name"` field. Show me how to keep only files whose names end in `.pdf`, and explain the list comprehension.

**Intermediate**

> Filter GitHub API file metadata so that only PDF files remain, using a case-insensitive filename check.

**Advanced**

> Suggest a robust way to filter GitHub Contents API results for PDF files while excluding directories and handling inconsistent filename capitalization.


In [ ]:
pdf_files = [
    file for file in files
    if file["name"].lower().endswith(".pdf")
]

[file["name"] for file in pdf_files]


## 3. Identify the direct URL of each PDF

The GitHub metadata contains a field called `download_url`.

This is the direct location of the raw PDF file.  
We collect those URLs into a list so that the **same extraction procedure** can be applied to every manifesto.

The conceptual move is:

**many PDF files → one list of PDF locations → one repeated procedure**

### Optional AI prompts

**Novice**

> Each item in my list `pdf_files` is a dictionary with a `"download_url"` field. How do I create a new list containing only those URLs?

**Intermediate**

> Extract the `download_url` field from every item in `pdf_files` using a Python list comprehension.

**Advanced**

> Given GitHub Contents API metadata, explain why `download_url` is preferable to constructing raw GitHub URLs manually. Mention possible limitations.


In [ ]:
pdf_urls = [file["download_url"] for file in pdf_files]

pdf_urls


## 4. Choose a library for reading PDF files

Python itself does not provide a convenient high-level tool for extracting text from PDFs.

For this notebook we use **PyMuPDF**, imported as `fitz`.

This is a good example of a situation where AI can be useful before coding: a student may know the task — *read PDF text in Python* — but may not know which library is appropriate.

In Google Colab, PyMuPDF may need to be installed once for the current session.

### Optional AI prompts

**Novice**

> I need to extract text from many PDF files in Google Colab. Which Python library would you recommend for a beginner? I need page-by-page text extraction and do not need OCR.

**Intermediate**

> Compare PyMuPDF, pypdf, and pdfplumber for extracting machine-readable text from PDFs in Python. Which would you choose for a simple corpus-extraction notebook?

**Advanced**

> I need a scalable PDF-extraction workflow. Explain when PyMuPDF text extraction is sufficient and when I would need OCR, layout-aware extraction, or another document-processing system.


In [ ]:
# Run this only if PyMuPDF is not already available in the environment.
# !pip install pymupdf


## 5. Read each PDF and extract its text

Now we perform the central extraction task.

For each PDF URL, the code:

1. reads the PDF content from GitHub into memory;
2. opens that content with PyMuPDF;
3. starts an empty string for the document;
4. visits every page in the PDF;
5. extracts the text from each page;
6. appends each page's text to the complete manifesto;
7. derives the party name from the PDF filename; and
8. stores the result in a Python dictionary.

The dictionary will have the structure:

```text
party → complete manifesto text
```

For example:

```text
BuenGobierno → "...complete extracted text..."
```

### Why page by page?

A PDF is internally organized as a collection of pages.  
PyMuPDF therefore exposes those pages individually. We visit each one and reconstruct the complete document text.

### Why a dictionary first?

At this stage, a dictionary gives us a very simple representation:

- **key** = party
- **value** = complete extracted manifesto

We will convert this into a table only after extraction is complete.

### Optional AI prompts

**Novice**

> I have a list of URLs pointing directly to PDF files. Using `requests` and PyMuPDF, write explicit beginner-friendly Python code that reads each PDF into memory, extracts the text page by page, gets the party name from the PDF filename, and stores the result in a dictionary called `documents`. Add comments explaining every step.

**Intermediate**

> Using `requests` and `pymupdf`, loop through `pdf_urls`, extract all page text from each remote PDF, derive the document identifier from the filename, and store the result as `{identifier: text}`. Keep the loop explicit rather than using a compact comprehension.

**Advanced**

> Review this PDF-extraction strategy for robustness. What checks would you add for HTTP errors, encrypted PDFs, empty page text, malformed PDFs, rate limits, or scanned documents? Keep the basic extraction logic unchanged.


In [ ]:
import pymupdf as fitz

# Create an empty dictionary.
# Each party will become a key, and its complete manifesto text will be the value.
documents = {}

# Go through the PDF URLs one at a time.
for pdf_url in pdf_urls:

    # Read the PDF file from its GitHub URL.
    # The PDF is kept in memory; we do not save a local copy.
    pdf_data = requests.get(pdf_url).content

    # Open the PDF content so PyMuPDF can read its pages.
    doc = fitz.open(stream=pdf_data, filetype="pdf")

    # Start with an empty string for this manifesto.
    text = ""

    # Visit every page and add its extracted text to the manifesto text.
    for page in doc:
        text += page.get_text()

    # Get the party name from the PDF filename.
    # Example: .../BuenGobierno.pdf → BuenGobierno
    party = pdf_url.split("/")[-1].replace(".pdf", "")

    # Store the complete manifesto using the party name as its key.
    documents[party] = text


## 6. Verify the extracted documents

Before transforming the extracted text into a DataFrame, we check the dictionary keys.

They should correspond to the party names derived from the PDF filenames.

This is a simple **validation step**:

> Did the extraction process produce one identifiable document for every manifesto we expected?

### Optional AI prompts

**Novice**

> I stored documents in a Python dictionary where the keys are party names. How can I inspect only the keys to verify which documents were extracted?

**Intermediate**

> Suggest two simple checks to confirm that a dictionary of extracted documents contains the expected files before I convert it to a DataFrame.

**Advanced**

> Propose lightweight extraction-validation checks that could detect missing documents, duplicated identifiers, or unexpectedly empty text without introducing a full testing framework.


In [ ]:
documents.keys()


## 7. Convert the dictionary into a corpus

The dictionary was convenient for extraction, but most later analysis will be easier with a rectangular data structure.

We therefore convert `documents` into a Pandas DataFrame called `corpus`.

Our **unit of observation** is the manifesto:

> **One row = one party manifesto**

The first version of the corpus contains two variables:

- `party`: the party name taken from the PDF filename;
- `text_raw`: the complete text extracted from that party's PDF.

The suffix `_raw` is important.  
The documents have been extracted and organized, but their text has **not been cleaned**.

### Optional AI prompts

**Novice**

> I have a Python dictionary where each key is a party and each value is the complete text of its manifesto. Show me how to convert it into a Pandas DataFrame with columns `party` and `text_raw`. I want one manifesto per row.

**Intermediate**

> Convert a `{party: text}` dictionary into a Pandas DataFrame with one row per party and columns `party` and `text_raw`, using `DataFrame.from_dict`.

**Advanced**

> Compare two ways to convert a `{document_id: text}` dictionary into a tidy Pandas corpus. Explain why one-row-per-document is useful for later text analysis.


In [ ]:
import pandas as pd

# Convert the dictionary into a DataFrame:
# one row = one party manifesto
# one text_raw value = the complete extracted text of that manifesto
corpus = (
    pd.DataFrame.from_dict(
        documents,
        orient="index",
        columns=["text_raw"]
    )
    .reset_index(names="party")
)

corpus


## 8. Add simple document characteristics

Before cleaning the text, we can describe the **size of each raw document**.

We add:

- `n_characters`: number of characters in the extracted manifesto;
- `n_words`: approximate number of words, obtained by splitting the raw text at whitespace.

These measures are useful for two reasons:

1. they let us compare the size of the manifestos;
2. unusually small values may reveal an extraction problem.

> `n_words` is only a **simple descriptive word count**.  
> It is **not** the linguistic tokenization that will be used later for text analysis.

### Optional AI prompts

**Novice**

> My Pandas DataFrame has one document per row in a column called `text_raw`. Show me how to add the number of characters and a simple whitespace-based word count for each document. Explain why this is not the same as tokenization.

**Intermediate**

> Add `n_characters` and `n_words` to a Pandas corpus using vectorized string methods. Use whitespace splitting for the approximate word count.

**Advanced**

> For a raw text corpus, compare character count, whitespace word count, and tokenizer-based token count as document-length diagnostics. Which should I use before cleaning and why?


In [ ]:
# Count the number of characters in each manifesto
corpus["n_characters"] = corpus["text_raw"].str.len()

# Count words using whitespace as the separator
# This is a simple descriptive count, not linguistic tokenization
corpus["n_words"] = corpus["text_raw"].str.split().str.len()


### Inspect the document-level summary

We do not need to print the full manifesto text to inspect the corpus.

A compact summary of party, characters, and words is enough to see whether the documents have plausible sizes.


In [ ]:
corpus[["party", "n_characters", "n_words"]]


## 9. Save the raw corpus

The extraction stage is complete.

We save the corpus as a CSV file so later notebooks do not need to repeat the PDF-extraction process.

The pipeline is now:

```text
GitHub RAW PDFs
      ↓
discover files
      ↓
read PDFs
      ↓
extract page text
      ↓
documents dictionary
      ↓
corpus DataFrame
      ↓
corpus_raw.csv
```

`corpus_raw.csv` is an **intermediate data product**:

- the PDFs have been converted into machine-readable text;
- the documents have been organized into one row per manifesto;
- basic size variables have been added;
- the textual content itself remains raw.

The file can now be uploaded to the repository's `preprocessed` folder and used as the starting point of the next notebook.

### Optional AI prompts

**Novice**

> I have a Pandas DataFrame called `corpus`. Show me how to save it as `corpus_raw.csv` without saving the DataFrame index.

**Intermediate**

> Save a Pandas corpus to CSV as `corpus_raw.csv` with one row per document and no index column.

**Advanced**

> I am building a reproducible text-analysis pipeline. Compare CSV and Parquet for storing a corpus containing document IDs, long raw-text strings, and numeric metadata. What are the trade-offs for teaching, GitHub, Python, and R interoperability?


In [ ]:
corpus.to_csv("corpus_raw.csv", index=False)


# What have we accomplished?

We started with PDF files designed primarily for human reading and transformed them into a computational corpus.

We have **not** yet:

- converted text to lowercase;
- removed punctuation;
- removed stopwords;
- tokenized the documents for analysis;
- normalized words;
- applied stemming or lemmatization;
- created n-grams;
- represented documents numerically.

Those decisions belong to later stages of the text-analytics journey.

For now, the goal was deliberately narrower:

> **Extract first. Preserve the raw text. Organize it. Verify it.**

---

## Reflection: how much AI assistance did you need?

The prompts above deliberately range from novice to advanced.

A useful exercise is to compare your own workflow:

- Which steps could you code without AI?
- Where did AI help you identify an unfamiliar library or API?
- Where did AI simply save time?
- Which suggestions still required you to verify the result?
- Would a more advanced user ask for less code and more methodological alternatives?

The objective is not to maximize AI use.  
It is to learn **where AI assistance is useful and how the quality of the prompt depends on your own level of computational understanding**.
